# StealthRL — Train Humanizer with DeBERTa Reward

RL fine-tunes a paraphraser (Qwen2.5-3B) so its output fools our DeBERTa detector.

Based on [StealthRL (arXiv 2602.08934)](https://arxiv.org/abs/2602.08934):
- **GRPO** (Group Relative Policy Optimization) with LoRA
- **Reward** = `1.0 * (1 - P_ai) + 0.3 * cosine_sim` (DeBERTa evasion + semantic preservation)
- **Data**: 5K AI texts from dataset.jsonl

**Requirements:**
1. Upload `detector_model/` to Google Drive at `MyDrive/ai-text-detector/detector_model/`
2. Upload `dataset.jsonl` to `MyDrive/ai-text-detector/dataset.jsonl`
3. Runtime → T4 GPU (free tier works)
4. Run all → ~2-3 hours
5. Download `stealth_lora/` adapter (~50MB) for local MLX inference

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl transformers datasets sentence-transformers safetensors
!pip install -q vllm  # required for Unsloth fast_inference
from unsloth import is_bfloat16_supported
print("Unsloth ready, bf16:", is_bfloat16_supported())

In [ ]:
# Cell 2: Mount Drive + load paths
from google.colab import drive
drive.mount('/content/drive')

import os, json, torch
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/ai-text-detector'
DETECTOR_DIR = os.path.join(DRIVE_DIR, 'detector_model')
DATASET_FILE = os.path.join(DRIVE_DIR, 'dataset.jsonl')
OUTPUT_DIR = os.path.join(DRIVE_DIR, 'stealth_lora')

assert os.path.exists(DETECTOR_DIR), f"DeBERTa not found at {DETECTOR_DIR}"
assert os.path.exists(DATASET_FILE), f"Dataset not found at {DATASET_FILE}"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
print(f"Detector: {DETECTOR_DIR}")
print(f"Dataset: {DATASET_FILE}")

In [ ]:
# Cell 3: Load paraphraser model + LoRA
from unsloth import FastLanguageModel

LORA_RANK = 32
MAX_SEQ = 1024  # A100 has plenty of VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    fast_inference=True,   # A100 has vLLM
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,  # leave room for DeBERTa + E5
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Paraphraser loaded: Qwen2.5-3B-Instruct + LoRA r={LORA_RANK}")

In [ ]:
# Cell 4: Load DeBERTa detector + E5 similarity model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# DeBERTa detector (our 98.5% model)
det_tokenizer = AutoTokenizer.from_pretrained(DETECTOR_DIR)
det_model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_DIR)
det_model.float().requires_grad_(False)
if torch.cuda.is_available():
    det_model = det_model.cuda()

# E5 for semantic similarity
e5_model = SentenceTransformer("intfloat/e5-base-v2")
if torch.cuda.is_available():
    e5_model = e5_model.to(torch.device("cuda"))

# Quick test
test_input = det_tokenizer("This is a test.", return_tensors="pt", truncation=True, max_length=512)
if torch.cuda.is_available():
    test_input = {k: v.cuda() for k, v in test_input.items()}
with torch.no_grad():
    test_probs = torch.softmax(det_model(**test_input).logits, dim=-1).cpu().numpy()[0]
print(f"DeBERTa test: human={test_probs[0]:.3f} ai={test_probs[1]:.3f} ap={test_probs[2]:.3f} hp={test_probs[3]:.3f}")
print(f"E5 loaded: {e5_model.get_sentence_embedding_dimension()}d")

In [ ]:
# Cell 5: Prepare training dataset — 5K AI texts from dataset.jsonl
from datasets import Dataset
import random

random.seed(42)
ai_texts = []
with open(DATASET_FILE) as f:
    for line in f:
        d = json.loads(line)
        # label 1 = pure AI, label 2 = AI polished — both are AI-authored
        if d["label"] in (1, 2):
            text = d["text"].strip()
            # 100-500 tokens range (approximate by words)
            word_count = len(text.split())
            if 30 <= word_count <= 300:
                ai_texts.append(text)

random.shuffle(ai_texts)
N_TRAIN = 5000
ai_texts = ai_texts[:N_TRAIN]
print(f"Training samples: {len(ai_texts)}")
print(f"Avg length: {np.mean([len(t.split()) for t in ai_texts]):.0f} words")

# Format as conversational prompts for GRPO
SYSTEM_MSG = "You are a writing assistant. Paraphrase the following text while preserving its meaning. Write naturally."

train_data = []
for text in ai_texts:
    train_data.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": text},
        ],
        # Store original text for semantic similarity reward
        "original_text": text,
    })

train_dataset = Dataset.from_list(train_data)
print(f"Dataset ready: {len(train_dataset)} samples")
print(f"Sample prompt: {train_data[0]['prompt'][1]['content'][:100]}...")

In [ ]:
# Cell 6: Define reward function (v2 — stepped reward + length penalty)
# v1 问题：线性 reward 区分度太低（DeBERTa 给所有 AI 文本 99-100%，导致 reward_std ~0.01）
# v2 修复：阶梯式 detector reward + 短文本惩罚

BETA = 0.3    # semantic preservation weight
SIM_GATE = 0.5
MIN_WORDS = 20  # 短于此的直接罚

def stealth_reward(prompts, completions, original_text, **kwargs):
    """GRPO reward: stepped detector evasion + semantic preservation + length gate."""
    rewards = []

    if isinstance(completions[0], list):
        comp_texts = [c[-1]["content"] if c else "" for c in completions]
    else:
        comp_texts = [str(c) for c in completions]

    if isinstance(original_text, list):
        orig_texts = original_text
    else:
        orig_texts = [original_text] * len(comp_texts)

    # Batch DeBERTa inference
    with torch.no_grad():
        inputs = det_tokenizer(
            comp_texts, truncation=True, max_length=512,
            padding=True, return_tensors="pt"
        )
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        logits = det_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    # Batch E5 similarity
    orig_embs = e5_model.encode(
        ["query: " + t for t in orig_texts],
        normalize_embeddings=True, show_progress_bar=False
    )
    comp_embs = e5_model.encode(
        ["query: " + t for t in comp_texts],
        normalize_embeddings=True, show_progress_bar=False
    )

    for i in range(len(comp_texts)):
        # Length gate: too short = truncated junk
        if len(comp_texts[i].split()) < MIN_WORDS:
            rewards.append(-0.5)
            continue

        # Semantic gate
        r_sem = float(np.dot(orig_embs[i], comp_embs[i]))
        if r_sem < SIM_GATE:
            rewards.append(-1.0)
            continue

        # Stepped detector reward (more gradient signal than linear)
        p_ai = float(probs[i][1] + probs[i][2])
        if p_ai < 0.3:
            r_det = 1.0   # fooled DeBERTa
        elif p_ai < 0.5:
            r_det = 0.6   # close
        elif p_ai < 0.7:
            r_det = 0.3   # some progress
        elif p_ai < 0.9:
            r_det = 0.1   # slight signal
        else:
            r_det = 0.0   # no progress

        reward = r_det + BETA * r_sem
        rewards.append(reward)

    return rewards

# Quick test
test_rewards = stealth_reward(
    prompts=["test"],
    completions=["This is a test sentence about artificial intelligence and how it changes the world today."],
    original_text=["Artificial intelligence is transforming the world in many different ways."]
)
print(f"Test reward: {test_rewards[0]:.3f}")

In [ ]:
# Cell 7: GRPO Training (v2 — A100 optimized + stepped reward)
from trl import GRPOConfig, GRPOTrainer

if gpu_mem > 60:  # A100-80GB
    num_gens = 8
    max_comp = 512
    max_prompt = 400
    batch = 4
    grad_accum = 1
elif gpu_mem > 30:  # A100-40GB
    num_gens = 6
    max_comp = 400
    max_prompt = 300
    batch = 2
    grad_accum = 2
else:  # T4 16GB
    num_gens = 2
    max_comp = 400
    max_prompt = 300
    batch = 1
    grad_accum = 4

MAX_STEPS = 500

training_args = GRPOConfig(
    use_vllm=True if gpu_mem > 30 else False,
    learning_rate=2e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=5,
    per_device_train_batch_size=batch,
    gradient_accumulation_steps=grad_accum,
    num_generations=num_gens,
    max_prompt_length=max_prompt,
    max_completion_length=max_comp,
    max_steps=MAX_STEPS,
    save_steps=100,
    max_grad_norm=0.1,
    beta=0.05,
    temperature=1.0,
    report_to="none",
    output_dir="/content/stealth_checkpoints",
)

print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
print(f"Config: gens={num_gens}, comp={max_comp}, prompt={max_prompt}, batch={batch}, accum={grad_accum}")
print(f"Effective batch: {batch * grad_accum}, Steps: {MAX_STEPS}, LR: {training_args.learning_rate}")
print(f"vLLM: {training_args.use_vllm}, KL: {training_args.beta}")

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[stealth_reward],
    processing_class=tokenizer,
)

print("\nStarting GRPO training (v2)...")
train_result = trainer.train()
print(f"\nTraining complete! Loss: {train_result.training_loss:.4f}")

In [ ]:
# Cell 8: Test the trained model before saving
print("=== Post-training evaluation ===\n")

test_ai_texts = [
    "Artificial intelligence has fundamentally transformed the landscape of modern education. Furthermore, the integration of machine learning algorithms into educational platforms has enabled personalized learning experiences that cater to individual student needs.",
    "Climate change is one of the most pressing challenges facing humanity today. The rising global temperatures have led to unprecedented weather events, including devastating hurricanes and prolonged droughts.",
    "The transformer architecture has revolutionized natural language processing by introducing the self-attention mechanism. Unlike recurrent neural networks, transformers can process entire sequences in parallel.",
]

FastLanguageModel.for_inference(model)

for i, orig in enumerate(test_ai_texts):
    # Generate paraphrase
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": orig},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, temperature=1.0, top_p=0.9, do_sample=True)
    paraphrase = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Score with DeBERTa
    det_input = det_tokenizer(paraphrase, truncation=True, max_length=512, return_tensors="pt")
    if torch.cuda.is_available():
        det_input = {k: v.cuda() for k, v in det_input.items()}
    with torch.no_grad():
        probs = torch.softmax(det_model(**det_input).logits, dim=-1).cpu().numpy()[0]
    ai_score = (probs[1] + probs[2]) * 100

    # Semantic similarity
    orig_emb = e5_model.encode(["query: " + orig], normalize_embeddings=True)
    para_emb = e5_model.encode(["query: " + paraphrase], normalize_embeddings=True)
    sim = float(np.dot(orig_emb[0], para_emb[0]))

    verdict = "HUMAN" if ai_score < 50 else "AI"
    status = "PASS" if verdict == "HUMAN" else "FAIL"

    print(f"Text {i+1}: AI={ai_score:.1f}% [{status}]  Sim={sim:.3f}")
    print(f"  Original:   {orig[:80]}...")
    print(f"  Paraphrase: {paraphrase[:80]}...")
    print()

In [ ]:
# Cell 9: Save LoRA adapter to Google Drive
import shutil, subprocess

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save LoRA adapter (not full model — just the adapter weights ~50MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Zip for download
zip_path = os.path.join(DRIVE_DIR, 'stealth_lora.zip')
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(['zip', '-r', zip_path, 'stealth_lora/'], cwd=DRIVE_DIR, check=True)

size = os.path.getsize(zip_path) / (1024*1024)
print(f"LoRA adapter saved: {OUTPUT_DIR}/")
print(f"Zip: {zip_path} ({size:.0f} MB)")

# List contents
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    s = os.path.getsize(p) / (1024*1024) if os.path.isfile(p) else 0
    print(f"  {f}: {s:.1f} MB")

print(f"\nTo use locally:")
print(f"1. Download stealth_lora.zip from Drive")
print(f"2. Unzip into ai-text-detector/models/stealth_lora/")
print(f"3. Load in MLX with adapter merge")